# Работа с API NASA

## Загрузка библиотек

In [25]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import json

## Получение данных

In [30]:
API_KEY = "XA0MqMwa4NiEjN69YZgq0wdBPKoiWqxldAxY42tz" # сгенерировать на сайте

df_ = pd.DataFrame(columns=['id', 'name', 'absolute_magnitude_h', 'estimated_diameter_min', 
'estimated_diameter_max', 'is_potentially_hazardous_asteroid','close_approach_date_full', 
'kilometers_per_hour', 'miss_distance', 'is_sentry_object' ])

In [23]:
def get_data(date, dataframe, api_key):
    """
    Получение данных за один день с API NASA
    """

    date = date.strftime('%Y-%m-%d')

    # URL эндпоинта
    url = f"https://api.nasa.gov/neo/rest/v1/feed"
    # Параметры для GET запроса
    params = {
        'start_date' : date,
        'end_date': date,
        'api_key': api_key
    } 

    # Отправка GET запроса для получения данных за один день
    response = requests.get(url, params=params)


    # Если код статуса ответа == 200, то считываем данные с json и формируем датасет
    if response.status_code == 200:
        data = response.text
        json_data = json.loads(data)

        # Итеративно проходим по каждому космическому объекту за выбранную дату
        for item in json_data['near_earth_objects'][date]:
            id = item['id']
            name = item['name']
            absolute_magnitude_h = item['absolute_magnitude_h']
            estimated_diameter_min = item['estimated_diameter']['meters']['estimated_diameter_min']
            estimated_diameter_max = item['estimated_diameter']['meters']['estimated_diameter_max']
            is_potentially_hazardous_asteroid = item['is_potentially_hazardous_asteroid']
            close_approach_date_full = item['close_approach_data'][0]['close_approach_date_full']
            kilometers_per_hour = item['close_approach_data'][0]['relative_velocity']['kilometers_per_hour']
            miss_distance = item['close_approach_data'][0]['miss_distance']['kilometers']
            is_sentry_object = item['is_sentry_object']

            new_row = {
                'id': id, 
                'name': name, 
                'absolute_magnitude_h':absolute_magnitude_h, 
                'estimated_diameter_min':estimated_diameter_min,
                'estimated_diameter_max':estimated_diameter_max,
                'is_potentially_hazardous_asteroid':is_potentially_hazardous_asteroid,
                'close_approach_date_full':close_approach_date_full,
                'kilometers_per_hour':kilometers_per_hour,
                'miss_distance':miss_distance,
                'is_sentry_object':is_sentry_object
            }

            dataframe.loc[-1] = new_row

        dataframe['close_approach_date_full'] = pd.to_datetime(dataframe['close_approach_date_full'])
        dataframe = dataframe.sort_values(by = 'close_approach_date_full').reset_index(drop=True)
        
        return dataframe # В качестве выходных данных функции мы возвращаем фрейм данных, содержащий результаты, полученные из API.

    else:
        print("Error:", response.status_code)

In [20]:
def get_data_with_ranges(start_date, end_date, df, api_key):
    
    # Формируем из строк даты
    start_date = datetime.strptime(start_date, '%Y-%m-%d')
    end_date = datetime.strptime(end_date, '%Y-%m-%d')

    # Итеративно проходим по каждому дню
    while start_date <= end_date:
        df = get_data(start_date, df, api_key) # Обновление датасета с добавлением данных за выбранный день
        start_date += timedelta(days=1)
    
    return df

In [31]:
df = get_data_with_ranges('2024-01-01','2024-02-01',df_, API_KEY)

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   id                                 32 non-null     object        
 1   name                               32 non-null     object        
 2   absolute_magnitude_h               32 non-null     float64       
 3   estimated_diameter_min             32 non-null     float64       
 4   estimated_diameter_max             32 non-null     float64       
 5   is_potentially_hazardous_asteroid  32 non-null     bool          
 6   close_approach_date_full           32 non-null     datetime64[ns]
 7   kilometers_per_hour                32 non-null     object        
 8   miss_distance                      32 non-null     object        
 9   is_sentry_object                   32 non-null     bool          
dtypes: bool(2), datetime64[ns](1), float64(3